# Colab: PPL Data-Augmentation Preview

Implements the **Phase 1 per-sample augmentations** from
[`AUGMENTATION_PLAN.md`](AUGMENTATION_PLAN.md) for the carbonate thin-section
segmenter, and previews them so you can visually confirm the masks track the
image and the results look realistic.

All imagery is **plane-polarized (PPL)**, so color is only weakly diagnostic —
photometric augmentation (color jitter, RGB shift, random grayscale, CLAHE,
gamma) is pushed harder, while the real signal (texture/relief/morphology) is
preserved. Geometric augmentations transform image **and** mask jointly with
nearest-neighbour interpolation and an `ignore_index` border fill.

Built on **torchvision `transforms.v2.functional`** + **cv2** (for CLAHE) — no
new dependency beyond what the pipeline already uses.

The final cell generates **10 augmentations of one sample** and displays them.


In [ ]:
# 1) Install dependencies (transformers/tqdm are needed to import the pipeline module)
!pip -q install --upgrade transformers tqdm

In [ ]:
# 2) Mount Google Drive (skip if running from a local clone)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not in Colab — will use the local repo clone.')

In [ ]:
# 3) Configure repo + dataset paths
import os, sys
from pathlib import Path

# Path to the cloned repo (Drive clone in Colab). Adjust if yours differs.
REPO_ROOT = Path("/content/drive/My Drive/Payne_lab_swin_transformer")
if not REPO_ROOT.is_dir():
    # Fallback: assume this notebook sits inside the repo (local run).
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "code").is_dir():
        REPO_ROOT = REPO_ROOT.parent

PIPELINE_DIR = REPO_ROOT / "code" / "model_training_pipeline"
sys.path.insert(0, str(PIPELINE_DIR))
os.chdir(PIPELINE_DIR)
print("REPO_ROOT  =", REPO_ROOT)

# Labeled image/mask dirs (Drive). Falls back to the repo sample data if absent.
IMG_DIR = Path("/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelledDataset_02032026/my_dataset/img")
MASK_DIR = Path("/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelledDataset_02032026/my_dataset/masks_machine")
if not IMG_DIR.is_dir() or not MASK_DIR.is_dir():
    IMG_DIR = REPO_ROOT / "data" / "carbonate_imgs_and_masks" / "img"
    MASK_DIR = REPO_ROOT / "data" / "carbonate_imgs_and_masks" / "masks"
    print("Using repo sample data.")
print("IMG_DIR    =", IMG_DIR)
print("MASK_DIR   =", MASK_DIR)

In [ ]:
# 3b) Update the repo clone so the pipeline module is current.
# This notebook needs the 18-class reconciliation (NUM_CLASSES=18, the
# `ignore_class_ids` dataset arg, ARTIFACT_CLASS_IDS) that was merged to main.
# Make sure the clone is on a branch that contains it (main or this branch).
import subprocess

def _git(*a):
    r = subprocess.run(["git", "-C", str(REPO_ROOT), *a], capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

print("branch:", _git("rev-parse", "--abbrev-ref", "HEAD"))
print(_git("fetch", "origin"))
print(_git("pull", "--ff-only") or "(nothing to pull / not a fast-forward)")
# If you are on an old branch, switch first, e.g.:
#   !git -C "$REPO_ROOT" checkout feat/augmentation-preview && git -C "$REPO_ROOT" pull

In [ ]:
# 4) Imports + reuse pipeline constants/dataset (single source of truth)
import math, random
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from torchvision.io import read_image
from torchvision.transforms.v2 import functional as TF
from torchvision.transforms.v2 import InterpolationMode

from swin_training_pipeline_221 import (
    CarbonateSegmentationDataset, CLASS_NAMES, NUM_CLASSES, IGNORE_INDEX,
    colorize_mask,
)
if NUM_CLASSES != 18:
    raise RuntimeError(
        f"Stale pipeline module: NUM_CLASSES={NUM_CLASSES} (expected 18). The repo clone "
        "predates the 18-class reconciliation. Run the '3b) Update repo' cell (checkout a "
        "branch with the 18-class changes, e.g. main / feat/augmentation-preview) and re-run."
    )
print("NUM_CLASSES =", NUM_CLASSES)

# Index pairs via the real dataset (transforms=None; we read raw uint8 ourselves below).
ds = CarbonateSegmentationDataset(
    root=str(IMG_DIR.parent),
    img_dir=str(IMG_DIR),
    mask_dir=str(MASK_DIR),
    transforms=None,
    normalize=False,
    strict=True,
    ignore_class_ids=(),  # keep every label id for the preview (artifacts shown too)
)
print("paired samples:", len(ds.pairs))


def read_raw_image(path):
    """uint8 [3,H,W] RGB."""
    im = read_image(str(path))
    if im.shape[0] == 1:
        im = im.repeat(3, 1, 1)
    elif im.shape[0] == 4:
        im = im[:3]
    return im.to(torch.uint8)


def read_raw_mask(path):
    """long [H,W] of class ids (0..17, plus 255 ignore)."""
    m = read_image(str(path))
    if m.ndim == 3 and m.shape[0] > 1:
        m = m[0:1, ...]
    return m.squeeze(0).to(torch.long)

In [ ]:
# 5) PPL augmentation library (torchvision v2 functional + cv2)
# Conventions enforced here:
#  - geometric ops transform image (bilinear, fill=0) AND mask (nearest, fill=IGNORE_INDEX)
#  - photometric ops touch the IMAGE ONLY (mask passes through unchanged)
#  - everything operates on uint8 CHW images, pre-normalization (so previews show true color)

DEFAULT_AUG = dict(
    crop=512,
    # rare-class-aware cropping: with prob rare_crop_prob, center the crop on a
    # randomly chosen rare-class pixel so small/rare grains actually appear.
    rare_crop_prob=0.5,
    rare_class_ids=(4, 6, 10, 12, 13, 14, 15, 16),  # echino, calc.algae, gastropod, mollusk, ostracod, agg.grain, brachiopod, sponge
    hflip_p=0.5, vflip_p=0.2,
    affine_p=0.7, affine_deg=15.0, affine_translate=0.05, affine_scale=(0.85, 1.15), affine_shear=8.0,
    # photometric (pushed harder because PPL color is non-diagnostic)
    color_jitter_p=0.9, brightness=0.30, contrast=0.30, saturation=0.40, hue=0.08,
    rgbshift_p=0.5, rgbshift_max=20,          # simulate microscope white-balance differences
    clahe_p=0.3, clahe_clip=2.0, clahe_grid=8,
    gamma_p=0.4, gamma_range=(0.7, 1.5),
    blur_p=0.2, blur_sigma=(0.1, 1.5),
    noise_p=0.3, noise_sigma=0.04,            # fraction of 255
    grayscale_p=0.10,                         # drop color -> learn from texture, not lighting
)


def _u8(t):
    return t.clamp(0, 255).to(torch.uint8)


# ----- geometric (image + mask jointly) -----
def rare_class_aware_crop(img, mask_c, cfg):
    _, H, W = img.shape
    crop = cfg["crop"]
    ch, cw = min(crop, H), min(crop, W)
    top = left = None
    label = "crop:random"
    rare = [c for c in cfg["rare_class_ids"] if bool((mask_c == c).any())]
    if rare and random.random() < cfg["rare_crop_prob"]:
        cid = random.choice(rare)
        ys, xs = torch.where(mask_c[0] == cid)
        j = random.randrange(len(ys))
        cy, cx = int(ys[j]), int(xs[j])
        top = min(max(cy - ch // 2, 0), max(H - ch, 0))
        left = min(max(cx - cw // 2, 0), max(W - cw, 0))
        label = f"crop:rare({CLASS_NAMES[cid]})"
    if top is None:
        top = random.randint(0, max(H - ch, 0))
        left = random.randint(0, max(W - cw, 0))
    img = TF.crop(img, top, left, ch, cw)
    mask_c = TF.crop(mask_c, top, left, ch, cw)
    if (ch, cw) != (crop, crop):  # smaller-than-crop image: resize up to crop
        img = TF.resize(img, [crop, crop], interpolation=InterpolationMode.BILINEAR, antialias=True)
        mask_c = TF.resize(mask_c, [crop, crop], interpolation=InterpolationMode.NEAREST)
        label += "+resize"
    return img, mask_c, label


def affine(img, mask_c, cfg):
    angle = random.uniform(-cfg["affine_deg"], cfg["affine_deg"])
    tx = int(random.uniform(-cfg["affine_translate"], cfg["affine_translate"]) * img.shape[-1])
    ty = int(random.uniform(-cfg["affine_translate"], cfg["affine_translate"]) * img.shape[-2])
    sc = random.uniform(*cfg["affine_scale"])
    sh = random.uniform(-cfg["affine_shear"], cfg["affine_shear"])
    img = TF.affine(img, angle=angle, translate=[tx, ty], scale=sc, shear=[sh],
                    interpolation=InterpolationMode.BILINEAR, fill=0)
    mask_c = TF.affine(mask_c, angle=angle, translate=[tx, ty], scale=sc, shear=[sh],
                       interpolation=InterpolationMode.NEAREST, fill=IGNORE_INDEX)
    label = f"affine(rot={angle:+.0f}deg, scale={sc:.2f}, shear={sh:+.0f}deg)"
    return img, mask_c, label


# ----- photometric (image only) -----
def color_jitter(img, cfg):
    b, c, s, h = cfg["brightness"], cfg["contrast"], cfg["saturation"], cfg["hue"]
    ops = [
        lambda x: TF.adjust_brightness(x, 1 + random.uniform(-b, b)),
        lambda x: TF.adjust_contrast(x, 1 + random.uniform(-c, c)),
        lambda x: TF.adjust_saturation(x, 1 + random.uniform(-s, s)),
        lambda x: TF.adjust_hue(x, random.uniform(-h, h)),
    ]
    random.shuffle(ops)
    for op in ops:
        img = op(img)
    return img


def rgb_shift(img, cfg):
    m = cfg["rgbshift_max"]
    shift = torch.randint(-m, m + 1, (3, 1, 1))
    s = shift.flatten().tolist()
    return _u8(img.to(torch.int16) + shift), f"rgb_shift({s[0]:+d},{s[1]:+d},{s[2]:+d})"


def clahe(img, cfg):
    hwc = img.permute(1, 2, 0).cpu().numpy()                       # HWC uint8 RGB
    lab = cv2.cvtColor(hwc, cv2.COLOR_RGB2LAB)
    cla = cv2.createCLAHE(clipLimit=cfg["clahe_clip"], tileGridSize=(cfg["clahe_grid"], cfg["clahe_grid"]))
    lab[:, :, 0] = cla.apply(lab[:, :, 0])
    rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).contiguous()


def gaussian_noise(img, cfg):
    noise = torch.randn(img.shape, dtype=torch.float32) * (cfg["noise_sigma"] * 255.0)
    return _u8(img.to(torch.float32) + noise)


def augment(img, mask, cfg=DEFAULT_AUG, return_ops=False):
    """img: uint8 [3,H,W]; mask: long [H,W].

    Returns (img_u8, mask_long), or (img_u8, mask_long, applied_ops) when
    return_ops=True. `applied_ops` is the list of augmentation labels that
    actually fired this call (the crop always runs).
    """
    mask_c = mask.unsqueeze(0)
    applied = []

    # --- geometric (image + mask) ---
    img, mask_c, lbl = rare_class_aware_crop(img, mask_c, cfg)
    applied.append(lbl)
    if random.random() < cfg["hflip_p"]:
        img, mask_c = TF.hflip(img), TF.hflip(mask_c)
        applied.append("hflip")
    if random.random() < cfg["vflip_p"]:
        img, mask_c = TF.vflip(img), TF.vflip(mask_c)
        applied.append("vflip")
    if random.random() < cfg["affine_p"]:
        img, mask_c, lbl = affine(img, mask_c, cfg)
        applied.append(lbl)

    # --- photometric (image only) ---
    if random.random() < cfg["color_jitter_p"]:
        img = color_jitter(img, cfg)
        applied.append("color_jitter")
    if random.random() < cfg["rgbshift_p"]:
        img, lbl = rgb_shift(img, cfg)
        applied.append(lbl)
    if random.random() < cfg["clahe_p"]:
        img = clahe(img, cfg)
        applied.append("clahe")
    if random.random() < cfg["gamma_p"]:
        g = random.uniform(*cfg["gamma_range"])
        img = TF.adjust_gamma(img, g)
        applied.append(f"gamma({g:.2f})")
    if random.random() < cfg["blur_p"]:
        s = random.uniform(*cfg["blur_sigma"])
        img = TF.gaussian_blur(img, kernel_size=5, sigma=s)
        applied.append(f"blur(sigma={s:.1f})")
    if random.random() < cfg["noise_p"]:
        img = gaussian_noise(img, cfg)
        applied.append("noise")
    if random.random() < cfg["grayscale_p"]:
        img = TF.rgb_to_grayscale(img, num_output_channels=3)
        applied.append("grayscale")

    img, mask = _u8(img), mask_c.squeeze(0)
    if return_ops:
        return img, mask, applied
    return img, mask

In [ ]:
# 6) Visualization helpers
import textwrap

# Fixed 18-colour palette so a class keeps the same colour across panels.
def make_palette(n):
    rng = np.random.default_rng(7)
    pal = [(int(rng.integers(40, 235)), int(rng.integers(40, 235)), int(rng.integers(40, 235))) for _ in range(n)]
    pal[0] = (35, 35, 35)  # background = dark grey
    return pal

PALETTE = make_palette(NUM_CLASSES)


def to_hwc(img_u8):
    return img_u8.permute(1, 2, 0).cpu().numpy().astype(np.uint8)


def overlay(img_u8, mask_long, alpha=0.5):
    base = to_hwc(img_u8)
    col = np.array(colorize_mask(mask_long.cpu().numpy().astype(np.uint8), PALETTE))
    ov = (alpha * col + (1 - alpha) * base).astype(np.uint8)
    return base, col, ov


def show_samples(samples, labels):
    """samples: list of (img_u8, mask_long); labels: per-row augmentation strings.

    Renders 3 columns per row: image (titled with the augmentations applied) |
    mask | overlay.
    """
    rows = len(samples)
    fig, axes = plt.subplots(rows, 3, figsize=(12, 4.6 * rows))
    if rows == 1:
        axes = axes[None, :]
    for r, (img, mask) in enumerate(samples):
        base, col, ov = overlay(img, mask)
        for c, panel in enumerate([base, col, ov]):
            axes[r, c].imshow(panel)
            axes[r, c].set_axis_off()
        axes[r, 0].set_title(textwrap.fill(labels[r], width=48), fontsize=8, loc="left")
        axes[r, 1].set_title("mask", fontsize=9)
        axes[r, 2].set_title("overlay", fontsize=9)
    plt.tight_layout()
    plt.show()

## Generate & preview 10 augmentations

The cell below takes one labeled sample and produces **10 random augmentations**,
showing each as *image | colorized mask | overlay* so you can confirm the mask
stays pixel-aligned with the augmented image. Change `SAMPLE_IDX` to inspect a
different image; rerun to draw new random augmentations.


In [ ]:
# 8) Create 10 augmentations of one sample and display them
SAMPLE_IDX = 0          # index into ds.pairs; change to preview a different image
N_AUG = 10
random.seed(0); torch.manual_seed(0)   # remove these two for fresh randomness each run

img0 = read_raw_image(ds.pairs[SAMPLE_IDX][0])
mask0 = read_raw_mask(ds.pairs[SAMPLE_IDX][1])

present = sorted(set(int(v) for v in torch.unique(mask0).tolist()) - {IGNORE_INDEX})
print("Sample", SAMPLE_IDX, "classes present:",
      [CLASS_NAMES[c] for c in present if c < NUM_CLASSES])

samples = [(img0, mask0)]
labels = ["original (no augmentation)"]
print("\nAugmentations applied per sample:")
for k in range(N_AUG):
    aug_img, aug_mask, ops = augment(img0.clone(), mask0.clone(), return_ops=True)
    samples.append((aug_img, aug_mask))
    label = f"aug {k + 1}: " + ", ".join(ops)
    labels.append(label)
    print("  " + label)

show_samples(samples, labels)